# RetractorDB storage from a notebook

Stage 1a of the embedded programme: the storage layer bound into CPython, read-only,
with no daemon and no query plan. See `docs/jupyter-integration.md` for what this
covers and what it deliberately does not.

**Prerequisite.** The tree must be configured with `-DRDB_PYTHON=ON` and built. The
cell below finds the module in the build tree; nothing needs to be installed.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[2]  # api/python/notebooks -> repo root

roots = [p for p in sorted((REPO / 'build').glob('*/python')) if any((p / 'retractordb').glob('_core*'))]
assert roots, 'no built extension found - configure with -DRDB_PYTHON=ON and build'
sys.path.insert(0, str(roots[0]))

import retractordb as rdb
print('module root:', roots[0])


## A descriptor

Any `.desc` file. Here we use a fixture that ships with the test suite: two BYTE
fields over four records.


In [ ]:
FIXTURE = REPO / 'test' / 'IntegrationTest' / 'issue153_storagemap_meta_cases'

desc = rdb.load_descriptor(str(FIXTURE / 'plain_file.desc'))
print(desc)
print('record width:', desc.size_bytes, 'bytes')
for field in desc:
    print(f'  {field.name:8} {field.type.name:8} offset={desc.byte_offset(field.name)}')


## Records

Open on a **copy** of the fixture. Opening a storage can write to its `.meta`
sidecar, and a versioned fixture should not be dirtied by a demo.


In [ ]:
import shutil, tempfile

workdir = Path(tempfile.mkdtemp())
for name in ('plain_file', 'plain_file.desc', 'plain_file.meta', 'plain_file.shadow'):
    shutil.copy2(FIXTURE / name, workdir / name)

with rdb.Storage('plain_file', 'plain_file', storage_param=str(workdir)) as storage:
    print(storage, 'declared source:', storage.is_declared)
    for record in (storage[i] for i in range(len(storage))):
        print(record.index, list(record))


Two things worth noticing in that output.

`None` is a real NULL, carried from the storage's `.meta` sidecar rather than
guessed from a sentinel value. And a negative index reads from the end, which is
how `revRead()` is spelled here:


In [ ]:
with rdb.Storage('plain_file', 'plain_file', storage_param=str(workdir)) as storage:
    print('last record:', list(storage[-1]))

    kept = storage[0]
    storage[1]                      # kept is a copy, so this does not disturb it
    print('still valid:', list(kept))


## What raises, and what does not

The binding guards the mistakes it can: a missing file, a missing directory, an
index out of range, a read from a DEVICE or TEXTSOURCE stream. Those raise normally
and your kernel survives.


In [ ]:
for attempt in (lambda: rdb.load_descriptor('/nonexistent/x.desc'),
                lambda: rdb.Storage('plain_file', 'plain_file', storage_param='/nonexistent'),
                lambda: rdb.Storage('', 'plain_file', storage_param=str(workdir))):
    try:
        attempt()
    except (rdb.RetractorDBError, ValueError) as exc:
        print(type(exc).__name__, '-', exc)


**What is not guarded, and cannot be yet.** A `.desc` file that exists but holds an
invalid descriptor reaches `FatalError`, which calls `std::exit`. That does not
unwind, so no `except` will see it - the kernel dies and the session goes with it.
There are 79 such sites in the storage layer alone.

Converting them to exceptions is phase 1 of the shared refactor, and this notebook
is the argument for doing that work against Python first: the same defect found
here costs a kernel restart, and found on a phone costs a crash report and a device
build. `api/python/tests/test_fatal_paths.py` pins the current behaviour so the
change is visible the moment it lands.

Do not run the next cell unless you want to see the kernel die - it is here to be
read, not executed.


In [ ]:
# malformed = workdir / 'broken.desc'
# malformed.write_text('not a descriptor')
# rdb.load_descriptor(str(malformed))      # kills the kernel, by design, for now
